In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname = "UNK",
    dt_ns          = 2.0,
    output_dir     = Path("./hbond_results"),
    figures_dir    = Path("./figures"),
)

# Dihedral definitions: (label, atom1, atom2, atom3, atom4)
# atom names must match residue name cfg.ligand_resname
DIHEDRAL_DEFS = [
    # ("C6-C7",   "N3",  "C6",  "C7",  "F1" ),
    # ("C8-C9",   "N3",  "C8",  "C9",  "C10"),
    # Add your dihedral definitions here
]

# For MDAnalysis backend (preferred — no PyMOL required)
TOPOLOGY  = Path("../run01/equilibrating_topology.pdb")
TRAJECTORY = Path("../run01/trajectory.xtc")

# For PyMOL backend (optional — needed if trajectory is PSE only)
PSE_PATH = Path("../run01/traj.pse")
USE_PYMOL = False   # set True to use PyMOL backend

# HBond events CSV from NB02 (for shading)
HBOND_EVENTS_CSV = cfg.output_dir / "hbond_all_events_run01.csv"

SAMPLE_NAME = "run01"
OUT_CSV = cfg.output_dir / f"dihedrals_{SAMPLE_NAME}.csv"

# Optional: time of conformational flip to draw a vertical marker
FLIP_TIME: float | None = None   # e.g. 132.0 (ns)
# ============================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mdatools.analysis.dihedral import DihedralAnalyzer

cfg.make_dirs()

analyzer = DihedralAnalyzer(cfg, DIHEDRAL_DEFS)

if OUT_CSV.exists():
    df = pd.read_csv(OUT_CSV)
    df["time_ns"] = df["frame"] * cfg.dt_ns
    print(f"Loaded existing CSV: {OUT_CSV}")
elif USE_PYMOL:
    df = analyzer.run_from_pse(PSE_PATH, OUT_CSV)
    print(f"Extracted via PyMOL: {OUT_CSV}")
else:
    from mdatools.universe import load_and_align
    u = load_and_align(TOPOLOGY, TRAJECTORY, cfg)
    df = analyzer.run(u)
    df.to_csv(OUT_CSV, index=False)
    print(f"Extracted via MDAnalysis: {OUT_CSV}")

print(df.head())

In [ ]:
# Load H-bond mask for shading
hbond_mask = None
if HBOND_EVENTS_CSV.exists():
    df_hb = pd.read_csv(HBOND_EVENTS_CSV)
    hbond_frames = set(df_hb["frame"].tolist())
    hbond_mask = np.array([f in hbond_frames for f in df["frame"]])

In [ ]:
from mdatools.plotting.dihedral_plots import plot_dihedral_timeseries, plot_dihedral_heatmap

dihedral_cols = [c for c in df.columns if c not in {"frame", "time_ns", "has_hbond"}]

if dihedral_cols:
    fig = plot_dihedral_timeseries(
        df,
        dihedral_cols=dihedral_cols,
        hbond_mask=hbond_mask,
        flip_time=FLIP_TIME,
        save_path=cfg.figures_dir / f"dihedral_timeseries_{SAMPLE_NAME}.png",
    )
    plt.show()

    fig2 = plot_dihedral_heatmap(
        df,
        dihedral_cols=dihedral_cols,
        save_path=cfg.figures_dir / f"dihedral_heatmap_{SAMPLE_NAME}.png",
    )
    plt.show()
else:
    print("No dihedral columns found. Add entries to DIHEDRAL_DEFS in the config cell.")